# InstructGPT — Training Language Models to Follow Instructions with Human Feedback (2022)

_Papers · Transformers & LLMs_

**Align a language model to human intent in three stages: fine-tune, learn a reward from human preferences, then optimize against it with reinforcement learning.**

---

This notebook is a practice scaffold for the **InstructGPT — Training Language Models to Follow Instructions with Human Feedback (2022)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math

torch.manual_seed(0)

# --- 0. Worked example: pairwise ranking loss for ONE comparison (Eqn. 1). ---
rw, rl = 2.0, 0.5
gap = rw - rl
sig = 1.0 / (1.0 + math.exp(-gap))
loss_we = -math.log(sig)
print("worked example: r(y_w)=%.1f r(y_l)=%.1f  gap=%.1f  sigma=%.4f  loss=%.4f" % (
      rw, rl, gap, sig, loss_we))
# worked example: r(y_w)=2.0 r(y_l)=0.5  gap=1.5  sigma=0.8176  loss=0.2014


# --- Toy world: an "output" is a 4-dim vector. A hidden "good direction" defines
#     the ground-truth human preference: higher dot-product = preferred. ---
D = 4
good_dir = torch.tensor([1.0, -1.0, 0.5, 0.0])     # true preference (unknown to the models)
def true_score(y): return y @ good_dir             # the human we are imitating

# --- STAGE 1 (SFT, conceptual): the reference policy is a standard normal. ---
sft_mu, std = torch.zeros(D), 1.0                  # pi^SFT

# --- STAGE 2: reward model r_theta, trained with the pairwise ranking loss (Eqn. 1). ---
class RewardModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(D, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, y): return self.net(y).squeeze(-1)

rm = RewardModel()
rm_opt = torch.optim.Adam(rm.parameters(), lr=0.01)

def rm_accuracy(n=4000):                            # fraction of held-out pairs ranked correctly
    a, b = torch.randn(n, D), torch.randn(n, D)
    win = (true_score(a) >= true_score(b)).unsqueeze(1)
    yw, yl = torch.where(win, a, b), torch.where(win, b, a)
    return (rm(yw) > rm(yl)).float().mean().item()

print("\nSTAGE 2 — reward model accuracy before training:", round(rm_accuracy(), 3))
for step in range(400):
    rm_opt.zero_grad()
    a, b = torch.randn(64, D), torch.randn(64, D)
    win = (true_score(a) >= true_score(b)).unsqueeze(1)
    yw, yl = torch.where(win, a, b), torch.where(win, b, a)   # (winner, loser) per pair
    loss = -F.logsigmoid(rm(yw) - rm(yl)).mean()             # Eqn. 1
    loss.backward(); rm_opt.step()
print("STAGE 2 — reward model accuracy after  training:", round(rm_accuracy(), 3))

# --- STAGE 3: optimize the policy (a Gaussian's mean mu) to maximize r_theta(y)
#     minus beta * KL(policy || SFT).  beta>0 keeps it near SFT; beta=0 is the ablation. ---
def kl_to_sft(mu): return ((mu - sft_mu) ** 2 / (2 * std ** 2)).sum()   # KL of two equal-std Gaussians
def preferred_rate(mu, n=6000):                                          # TRUE preferred-output rate
    ps = true_score((mu + std * torch.randn(n, D)).detach())
    ss = true_score(sft_mu + std * torch.randn(n, D))
    return (ps > ss).float().mean().item()

for beta, tag in [(1.0, "WITH KL penalty (beta=1.0)"), (0.0, "NO KL penalty (beta=0, ablation)")]:
    torch.manual_seed(1)
    mu = nn.Parameter(sft_mu.clone())                     # policy starts AT the SFT model
    p_opt = torch.optim.Adam([mu], lr=0.02)
    pref0 = preferred_rate(mu)
    for step in range(200):
        p_opt.zero_grad()
        y = mu + std * torch.randn(64, D)                 # policy samples (reward model is FROZEN)
        loss = -rm(y).mean() + beta * kl_to_sft(mu)       # Eqn. 2 (gamma=0): -reward + beta*KL
        loss.backward(); p_opt.step()
    print("\nSTAGE 3 — %s" % tag)
    print("  preferred-rate: %.3f -> %.3f" % (pref0, preferred_rate(mu)))
    print("  KL to SFT = %.1f   policy mu = %s" % (
          kl_to_sft(mu).item(), [round(v, 2) for v in mu.tolist()]))

# STAGE 2 — reward model accuracy before training: 0.646
# STAGE 2 — reward model accuracy after  training: 0.993
#
# STAGE 3 — WITH KL penalty (beta=1.0)
#   preferred-rate: 0.502 -> 1.000
#   KL to SFT = 11.9   policy mu = [3.0, -3.03, 2.37, 0.32]   <- 4th coord (true weight 0) barely moves
# STAGE 3 — NO KL penalty (beta=0, ablation)
#   preferred-rate: 0.502 -> 1.000
#   KL to SFT = 26.0   policy mu = [3.48, -3.49, 3.38, 4.04]  <- 4th coord DRIFTS to 4.04
# (Our small toy run, not the paper's reported numbers. The reward model is only an
#  estimate; without the KL penalty the policy drifts where the reward has no real signal.)

## Visualize the data & results

_As we optimize the policy against the learned reward, its preferred-output rate rises — but with NO KL penalty the policy keeps drifting away from the SFT model (over-optimization). The KL penalty reaches the same preference quality while staying far closer to SFT._

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math
torch.manual_seed(0)

D = 4
good_dir = torch.tensor([1.0, -1.0, 0.5, 0.0])
def true_score(y): return y @ good_dir

# --- STAGE 2: reward model trained with the pairwise ranking loss (Eqn. 1) ---
class RewardModel(nn.Module):
    def __init__(self):
        super().__init__(); self.net = nn.Sequential(nn.Linear(D,16), nn.ReLU(), nn.Linear(16,1))
    def forward(self, y): return self.net(y).squeeze(-1)
rm = RewardModel(); opt = torch.optim.Adam(rm.parameters(), lr=0.01)
def rm_acc(n=4000):
    a,b = torch.randn(n,D), torch.randn(n,D); win = (true_score(a)>=true_score(b)).unsqueeze(1)
    yw,yl = torch.where(win,a,b), torch.where(win,b,a)
    return (rm(yw) > rm(yl)).float().mean().item()
print("RM acc before:", round(rm_acc(),3))
for step in range(400):
    opt.zero_grad()
    a,b = torch.randn(64,D), torch.randn(64,D); win = (true_score(a)>=true_score(b)).unsqueeze(1)
    yw,yl = torch.where(win,a,b), torch.where(win,b,a)
    (-F.logsigmoid(rm(yw)-rm(yl)).mean()).backward(); opt.step()
print("RM acc after :", round(rm_acc(),3))

# --- STAGE 3: policy optimization, with vs without the KL penalty (Eqn. 2, gamma=0) ---
sft_mu, std = torch.zeros(D), 1.0
def kl_to_sft(mu): return ((mu-sft_mu)**2 / (2*std**2)).sum()
def pref_rate(mu, n=6000):
    ps = true_score((mu+std*torch.randn(n,D)).detach()); ss = true_score(sft_mu+std*torch.randn(n,D))
    return (ps > ss).float().mean().item()

for beta, tag in [(1.0,"WITH KL"), (0.0,"NO KL (ablation)")]:
    torch.manual_seed(1)
    mu = nn.Parameter(sft_mu.clone()); p_opt = torch.optim.Adam([mu], lr=0.02)
    kl_curve = [(0, round(kl_to_sft(mu).item(),2))]; pr = [(0, round(pref_rate(mu),3))]
    for step in range(1, 201):
        p_opt.zero_grad()
        y = mu + std*torch.randn(64,D)
        (-rm(y).mean() + beta*kl_to_sft(mu)).backward(); p_opt.step()
        if step % 40 == 0:
            kl_curve.append((step, round(kl_to_sft(mu).item(),2)))
            pr.append((step, round(pref_rate(mu),3)))
    print(tag, "final mu", [round(v,2) for v in mu.tolist()], "KL=%.1f"%kl_to_sft(mu).item())
    print("  KL curve  :", kl_curve)
    print("  pref-rate :", pr)
# RM acc before: 0.646
# RM acc after : 0.993
# WITH KL          final mu [3.0, -3.03, 2.37, 0.32] KL=11.9
#   KL curve  : [(0,0.0),(40,0.95),(80,3.01),(120,5.58),(160,8.58),(200,11.95)]
#   pref-rate : [(0,0.502),(40,0.824),(80,0.954),(120,0.987),(160,0.998),(200,1.0)]
# NO KL (ablation) final mu [3.48, -3.49, 3.38, 4.04] KL=26.0
#   KL curve  : [(0,0.0),(40,1.24),(80,4.49),(120,9.53),(160,16.65),(200,26.01)]
#   pref-rate : [(0,0.502),(40,0.826),(80,0.96),(120,0.992),(160,1.0),(200,1.0)]
# (Our small toy run, not the paper's number.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The KL ablation. You have a reward model trained on preferences and a policy optimized against it
            with KL coefficient $\beta \gt 0$. Set $\beta = 0$ and retrain the policy, everything else identical.
            What term did you just drop, and what do you expect to happen to (a) the policy's measured reward,
            (b) its distance from the SFT model, and (c) its true quality?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Locate the policy objective: maximize r(y) - BETA * KL(policy || sft). Set BETA = 0. — _With $\beta = 0$ the KL penalty vanishes, so nothing pulls the policy back toward $\pi^{\text{SFT}}$ &mdash; it can move anywhere that raises the reward._
- Identify the dropped term: the per-token KL penalty $-\beta\log(\pi_\phi^{\text{RL}}/\pi^{\text{SFT}})$ from Equation 2 is gone. — _That term was the only thing keeping the policy in the region where the reward model was trained and is trustworthy._
- Predict the outcome: measured reward goes up; KL distance to SFT grows large; true quality plateaus or drifts because the policy exploits directions the reward model never judged. — _The reward model is an approximation; off its training region a high score is a mirage (reward over-optimization)._

**Answer:** You dropped the KL penalty. (a) The measured reward typically rises &mdash; the
                 policy is now free to chase it without cost. (b) The distance from the SFT model grows large: in our
                 run the KL roughly doubles ($\approx 12$ with the penalty versus $\approx 26$ without). (c) But the
                 true preferred-rate does not improve further; the policy drifts in directions the reward model
                 cannot really judge (in our toy run, a coordinate the true preference does not care about wanders
                 from $\approx 0.3$ to $\approx 4.0$). This is reward over-optimization: the KL penalty is what keeps
                 the policy honest.

</details>

**Problem 2.** Why does the paper train the reward model on pairwise comparisons with the loss
            $-\log\sigma(r(y_w) - r(y_l))$, instead of asking labelers to assign each output an absolute quality
            score and regressing on it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Consider what humans are good at. Absolute scores ("rate this 7/10") are noisy and vary by labeler and by day; relative judgments ("A is better than B") are far more consistent. — _The loss only needs the ordering $r(y_w) \gt r(y_l)$, which matches the more reliable human signal._
- Look at the loss: it depends only on the score difference $r(y_w) - r(y_l)$, not on absolute values. — _Through the sigmoid, the difference becomes a preference probability (Bradley-Terry). The model is free to set the overall scale; only relative ordering is supervised._
- Note the data efficiency: one ranking of $K$ outputs yields $\binom{K}{2}$ pairwise comparisons (&sect;3.4). — _Ranking $K=4$ to $9$ outputs once gives many comparisons cheaply, so each labeling task produces a lot of training signal._

**Answer:** Because relative judgments are what humans give reliably. Absolute quality scores are noisy and
                 inconsistent across labelers; "A beats B" is stable. The ranking loss only uses the score
                 difference $r(y_w) - r(y_l)$, so it learns from orderings and lets the scale float &mdash;
                 and a single ranking of $K$ outputs produces $\binom{K}{2}$ comparisons, which is cheap and
                 efficient. The byproduct, the scalar $r_\theta$, then serves as the reward in Stage 3.

</details>

**Problem 3.** In the worked example the reward model scored the preferred output at $r(y_w) = 2.0$ and the
            dispreferred at $r(y_l) = 0.5$, giving loss $-\log\sigma(1.5) = 0.2014$. Suppose instead the reward
            model is more confident and correct: $r(y_w) = 3.0$, $r(y_l) = -1.0$. Compute the loss. Then
            suppose it is confidently wrong: $r(y_w) = -1.0$, $r(y_l) = 3.0$. Compute that loss. What does the
            comparison say?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Confident & correct: gap $= 3.0 - (-1.0) = 4.0$. $\sigma(4.0) = 1/(1+e^{-4}) = 0.9820$. Loss $= -\log(0.9820) = 0.0181$. — _A large positive gap makes $\sigma$ near $1$, so the loss is tiny &mdash; the model is barely corrected when it is right and confident._
- Confident & wrong: gap $= -1.0 - 3.0 = -4.0$. $\sigma(-4.0) = 0.0180$. Loss $= -\log(0.0180) = 4.017$. — _A large negative gap makes $\sigma$ near $0$, so $-\log$ is large &mdash; the model is strongly pushed to flip the ordering._
- Compare: $0.0181$ versus $4.017$ &mdash; a factor of about $220\times$. — _The loss is gentle when the ranking is right and steep when it is confidently wrong, which is exactly the gradient signal you want._

**Answer:** Confident-and-correct gives loss $\approx 0.0181$; confident-and-wrong gives loss $\approx 4.017$
                 &mdash; about $220\times$ larger. The pairwise ranking loss barely touches a correct, confident
                 ranking but punishes a confident mistake hard, so training drives the reward model to order
                 human-preferred outputs above the rest. (Contrast the original example's $0.2014$ for a modest
                 correct gap of $1.5$.)

</details>